# Out-of-distribution test: Norne

Load the XGBoost model trained on SPE9 (`xgboost_fpr.json`) and evaluate it on
`dataset_norne.csv`. The Norne dataset was generated with a different
benchmark (real Norwegian field, 113k cells, multi-PVT, 9-year history) and
has the same 16-column schema as the SPE9 dataset, with all dynamic columns
unit-converted to FIELD units (psi, STB, SCF, MSCF) so the schema matches.

This is a transferability test, not a benchmark of XGBoost itself. We expect
some degradation; the question is how much.


## 1. Setup


In [ ]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "xgboost>=2.0", "scikit-learn", "pandas", "matplotlib"], check=True)

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=UserWarning)


## 2. Load model and datasets


In [ ]:
FEATURES = [
    "Porosidad",
    "Permeabilidad_mD",
    "Presion_Burbuja_psi",
    "Caudal_Prod_Petroleo_bbl",
    "Caudal_Prod_Gas_Mpc",
    "Caudal_Iny_Agua_bbl",
    "Prod_Acumulada_Petroleo",
    "Prod_Acumulada_Gas",
    "Prod_Acumulada_Agua",
    "Iny_Acumulada_Agua",
]
TARGET = "Presion_Reservorio_psi"


def first_existing(*candidates: str) -> Path:
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f"none of {candidates} exist")


model_path = first_existing("xgboost_fpr.json", "../notebooks/xgboost_fpr.json", "notebooks/xgboost_fpr.json")
spe9_path = first_existing("datasets/dataset_spe9.csv", "../datasets/dataset_spe9.csv", "dataset.csv", "../dataset.csv")
norne_path = first_existing("datasets/dataset_norne.csv", "../datasets/dataset_norne.csv", "dataset_norne.csv", "../dataset_norne.csv")

model = xgb.XGBRegressor()
model.load_model(model_path)
print(f"loaded model: {model_path}")

spe9_df = pd.read_csv(spe9_path)
norne_df = pd.read_csv(norne_path)
print(f"SPE9 dataset:  {spe9_df.shape}")
print(f"Norne dataset: {norne_df.shape}")

## 3. Headline metrics on Norne

Compare against the SPE9 held-out reference: MAE 38 psi, R^2 0.988.


In [ ]:
X_norne = norne_df[FEATURES].to_numpy()
y_norne = norne_df[TARGET].to_numpy()

preds_norne = model.predict(X_norne)
mae_norne = mean_absolute_error(y_norne, preds_norne)
rmse_norne = float(np.sqrt(mean_squared_error(y_norne, preds_norne)))
r2_norne = r2_score(y_norne, preds_norne)

# Trivial baseline using SPE9 train median
spe9_median = spe9_df[TARGET].median()
baseline_pred = np.full_like(y_norne, spe9_median, dtype=float)
baseline_mae = mean_absolute_error(y_norne, baseline_pred)

results = pd.DataFrame(
    {
        "metric": ["MAE", "RMSE", "R^2", "MAE / mean(y)", "Baseline MAE (SPE9 median)"],
        "Norne": [
            f"{mae_norne:.2f} psi",
            f"{rmse_norne:.2f} psi",
            f"{r2_norne:.4f}",
            f"{mae_norne / y_norne.mean():.3%}",
            f"{baseline_mae:.2f} psi",
        ],
        "SPE9 reference": ["38.00 psi", "55.50 psi", "0.9882", "1.21%", "408.87 psi"],
    }
).set_index("metric")
results


## 4. Per-simulation metrics


In [ ]:
per_sim = []
for sid, grp in norne_df.groupby("sim_id"):
    p = model.predict(grp[FEATURES].to_numpy())
    per_sim.append(
        {
            "sim_id": sid,
            "MAE_psi": mean_absolute_error(grp[TARGET], p),
            "R2": r2_score(grp[TARGET], p),
            "FPR_min": grp[TARGET].min(),
            "FPR_max": grp[TARGET].max(),
        }
    )
per_sim_df = pd.DataFrame(per_sim).sort_values("MAE_psi")
print(per_sim_df.describe()[["MAE_psi", "R2"]].round(3))
print()
print("Best 3 sims:")
print(per_sim_df.head(3).round(3).to_string(index=False))
print()
print("Worst 3 sims:")
print(per_sim_df.tail(3).round(3).to_string(index=False))


## 5. Distribution shift diagnostic

For each feature used by the model, plot SPE9-train vs Norne. Features whose Norne range falls outside the SPE9-train range force the trees to extrapolate, which is where most of the error comes from.


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(13, 11))
axes = axes.flat
for ax, col in zip(axes, FEATURES):
    spe9_vals = spe9_df[col].values
    norne_vals = norne_df[col].values
    ax.hist(spe9_vals, bins=40, alpha=0.55, label="SPE9", color="steelblue", density=True)
    ax.hist(norne_vals, bins=40, alpha=0.55, label="Norne", color="firebrick", density=True)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)
    spe9_min, spe9_max = spe9_vals.min(), spe9_vals.max()
    norne_min, norne_max = norne_vals.min(), norne_vals.max()
    if norne_min < spe9_min or norne_max > spe9_max:
        ax.text(0.04, 0.86, "OUT OF RANGE", transform=ax.transAxes,
                fontsize=7, color="firebrick", weight="bold")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
for ax in axes[len(FEATURES):]:
    ax.set_visible(False)
fig.suptitle("Feature distributions: SPE9 (training) vs Norne (test)")
fig.tight_layout()
plt.show()


## 6. Predicted vs actual scatter


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
sc = ax.scatter(y_norne, preds_norne, c=norne_df["sim_id"], cmap="tab20", s=8, alpha=0.6)
lo = float(min(y_norne.min(), preds_norne.min()))
hi = float(max(y_norne.max(), preds_norne.max()))
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
ax.set_xlabel("FPR true (psi)")
ax.set_ylabel("FPR predicted (psi)")
ax.set_title("Predicted vs actual on Norne (full dataset)")
ax.grid(alpha=0.3)
ax.legend()
plt.colorbar(sc, ax=ax, label="sim_id")
plt.tight_layout()
plt.show()


## 7. Residuals


In [ ]:
residuals = y_norne - preds_norne
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(residuals, bins=40, color="slategray", edgecolor="black")
axes[0].axvline(0, color="firebrick")
axes[0].set_title(f"Residuals on Norne (mean={residuals.mean():+.0f}, std={residuals.std():.0f} psi)")
axes[0].set_xlabel("y_true - y_pred (psi)")
axes[0].grid(alpha=0.3)

axes[1].scatter(preds_norne, residuals, s=6, alpha=0.4)
axes[1].axhline(0, color="firebrick")
axes[1].set_xlabel("Predicted FPR (psi)")
axes[1].set_ylabel("Residual (psi)")
axes[1].set_title("Residuals vs predicted")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Per-sim traces

True vs predicted FPR over time for four representative Norne sims.


In [ ]:
sorted_sims = per_sim_df.sort_values("MAE_psi")
chosen = [
    sorted_sims.iloc[0]["sim_id"],
    sorted_sims.iloc[len(sorted_sims) // 3]["sim_id"],
    sorted_sims.iloc[2 * len(sorted_sims) // 3]["sim_id"],
    sorted_sims.iloc[-1]["sim_id"],
]

norne_df_step = norne_df.copy()
norne_df_step["step"] = norne_df_step.groupby("sim_id").cumcount()

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, sid in zip(axes.flat, chosen):
    sub = norne_df_step[norne_df_step["sim_id"] == sid].sort_values("step")
    p = model.predict(sub[FEATURES].to_numpy())
    ax.plot(sub["step"], sub[TARGET], color="firebrick", lw=1.5, label="true")
    ax.plot(sub["step"], p, color="steelblue", linestyle="--", lw=1.2, label="pred")
    sim_mae = mean_absolute_error(sub[TARGET], p)
    ax.set_title(f"Norne sim {int(sid):04d} (MAE = {sim_mae:.0f} psi)")
    ax.set_xlabel("Report step")
    ax.set_ylabel("FPR (psi)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("True vs predicted FPR over time, four Norne sims (best/median/p67/worst MAE)")
plt.tight_layout()
plt.show()


## 9. Notes on the result

How to read this notebook:

* **MAE on Norne** vs SPE9 reference 38 psi tells how badly the SPE9-trained
  model degrades on a benchmark with different geometry, well count, fluid
  properties, and operational ranges.
* **Distribution shift** plot is the diagnostic for *why* it degrades. If
  the cumulatives are flagged "OUT OF RANGE" (likely, given Norne's 9-year
  history vs SPE9's 900 days), the trees are extrapolating, which is the
  failure mode of choice for tree-based regressors. Bo, Bg and Rs are
  excluded from the model so their distributions are not relevant here.
* **Predicted vs actual scatter**: a tight diagonal band means generalization
  worked. Diagonal stripes by sim_id mean the model has the right *shape*
  per sim but the wrong *level*, which is partly correctable with simple
  re-calibration. Random scatter means the model picked up noise.
* **Per-sim traces**: confirm whether the model tracks the depletion
  dynamics over time or just predicts a flat number near the training
  mean.
